<p align="center">
  <img src="https://raw.githubusercontent.com/VK-Ant/sightrag/main/assets/sightrag_banner.png" alt="SightRAG" width="100%">
</p>

# SightRAG v0.3
### See. Search. Retrieve.

<p align="center">
  <a href="https://pypi.org/project/sightrag/"><img src="https://img.shields.io/pypi/v/sightrag" alt="PyPI"></a>
  <a href="https://github.com/VK-Ant/sightrag/blob/main/LICENSE"><img src="https://img.shields.io/badge/license-Apache%202.0-blue" alt="License"></a>
  <a href="https://www.python.org/"><img src="https://img.shields.io/badge/python-3.9+-green" alt="Python"></a>
</p>

**v0.3 features:** Grounding DINO (any domain) | Person Re-ID | CLI | Qdrant | Re-ranking

**GitHub:** [VK-Ant/sightrag](https://github.com/VK-Ant/sightrag) | **PyPI:** [pip install sightrag](https://pypi.org/project/sightrag/)

## Step 1 — Install

In [ ]:
!pip install sightrag -q

from sightrag import SightRAG, __version__
print(f'SightRAG v{__version__} installed')

## Step 2 — Create Demo Data

In [ ]:
from PIL import Image, ImageDraw
import os, matplotlib.pyplot as plt

for d in ['demo/input', 'demo/reference']:
    os.makedirs(d, exist_ok=True)

configs = [
    ('shelf_empty', (240,240,240), [(50,80,100,150,(255,0,0)), (120,80,170,150,(0,0,255))]),
    ('shelf_full', (240,240,240), [(50+i*70,80,100+i*70,150,(i*30,100,200)) for i in range(8)]),
    ('person_door', (200,200,220), [(250,100,310,160,(255,200,150)), (265,160,295,300,(0,0,180))]),
    ('person_stand', (180,220,180), [(280,80,360,160,(255,200,150)), (295,160,345,340,(200,0,0))]),
    ('parking', (80,80,80), [(50,280,120,430,(255,0,0)), (160,280,230,430,(0,0,255))]),
    ('room', (230,230,230), [(200,50,440,250,(135,206,250)), (450,250,550,350,(139,69,19))]),
]
for name, bg, rects in configs:
    img = Image.new('RGB', (640, 480), bg)
    draw = ImageDraw.Draw(img)
    if 'shelf' in name: draw.rectangle([20, 150, 620, 160], fill=(139, 69, 19))
    for r in rects: draw.rectangle(list(r[:4]), fill=r[4])
    img.save(f'demo/input/{name}.jpg')

img = Image.new('RGB', (200,300), (180,220,180))
draw = ImageDraw.Draw(img)
draw.ellipse([70,20,130,80], fill=(255,200,150))
draw.rectangle([80,80,120,200], fill=(200,0,0))
img.save('demo/reference/ref_person.jpg')

img = Image.new('RGB', (320,240), (240,240,240))
draw = ImageDraw.Draw(img)
draw.rectangle([10,80,310,88], fill=(139,69,19))
draw.rectangle([30,30,80,80], fill=(255,0,0))
img.save('demo/reference/ref_shelf.jpg')

print(f'Input: {len(os.listdir("demo/input"))} | Reference: {len(os.listdir("demo/reference"))}')

fig, axes = plt.subplots(1, 6, figsize=(18,3))
for ax, f in zip(axes, sorted(os.listdir('demo/input'))):
    ax.imshow(Image.open(f'demo/input/{f}')); ax.set_title(f.replace('.jpg',''), fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

## Step 3 — Index + Query (3 lines)

In [ ]:
rag = SightRAG()
rag.index('./demo/input/')
results = rag.query('find person')

for i, r in enumerate(results[:3], 1):
    print(f"{i}. {r['image_path'].split('/')[-1]} | score: {r['score']:.4f} | {r['label']}")

## Step 4 — Text + Reference Queries

In [ ]:
# Text queries
for q in ['find person', 'find car', 'find room', 'find shelf']:
    results = rag.query(q, top_k=2)
    print(f'\n"{q}"')
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['image_path'].split('/')[-1]} | score: {r['score']:.4f}")

# Reference queries
for ref in sorted(os.listdir('demo/reference')):
    results = rag.query(reference=f'demo/reference/{ref}', top_k=2)
    print(f'\nReference: {ref}')
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['image_path'].split('/')[-1]} | score: {r['score']:.4f}")

## Step 5 — Visualize with rag.show()

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

def visualize(query, results, max_show=3):
    n = min(len(results), max_show)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1: axes = [axes]
    for idx, r in enumerate(results[:n]):
        img = Image.open(r['image_path']).copy()
        draw = ImageDraw.Draw(img)
        bbox = r['bbox']
        if bbox and len(bbox) == 4:
            draw.rectangle(bbox, outline='red', width=3)
            draw.text((bbox[0]+2, bbox[1]-12), f"{r['label']} ({r['score']:.2f})", fill='red')
        axes[idx].imshow(img)
        axes[idx].set_title(f"{r['image_path'].split('/')[-1]}\nscore: {r['score']:.4f}", fontsize=10)
        axes[idx].axis('off')
    plt.suptitle(f'Query: "{query}"', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

visualize('find person', rag.query('find person', top_k=3))
visualize('find shelf', rag.query('find shelf', top_k=3))

# Save annotated images
os.makedirs('output', exist_ok=True)
rag.show(rag.query('find person', top_k=3), save='./output/')

## Step 6 — Grounding DINO (NEW in v0.3)
Detect ANY object by text description. No training needed.

In [ ]:
try:
    rag_dino = SightRAG(detector='grounding-dino')
    rag_dino.index('./demo/input/')
    
    for q in ['find person standing', 'find furniture', 'find colored rectangle']:
        results = rag_dino.query(q, top_k=2)
        print(f'\n"{q}"')
        for i, r in enumerate(results, 1):
            print(f"  {i}. {r['image_path'].split('/')[-1]} | score: {r['score']:.4f}")
    
    visualize('find person standing', rag_dino.query('find person standing', top_k=3))
    rag_dino.clear()
except Exception as e:
    print(f'Grounding DINO not available: {str(e)[:80]}')
    print('Install: pip install transformers torch')

## Step 7 — Re-ranking (NEW in v0.3)
Better results on large datasets.

In [ ]:
# Without re-ranking
results_normal = rag.query('find person', top_k=3)
print('Without re-ranking:')
for i, r in enumerate(results_normal, 1):
    print(f"  {i}. {r['image_path'].split('/')[-1]} | score: {r['score']:.4f}")

# With re-ranking
rag_rerank = SightRAG(rerank=True)
rag_rerank.index('./demo/input/')
results_reranked = rag_rerank.query('find person', top_k=3)
print('\nWith re-ranking:')
for i, r in enumerate(results_reranked, 1):
    print(f"  {i}. {r['image_path'].split('/')[-1]} | score: {r['score']:.4f}")
rag_rerank.clear()

## Step 8 — Backend Info

In [ ]:
print(f'SightRAG: {rag}')
print(f'Backend:  {rag._backend.name}')
print(f'Version:  {__version__}')
print()
print('Install ONNX for 2x speed:')
print('  !pip install onnxruntime')
print('  rag = SightRAG()  # auto-selects ONNX')

## Cleanup

In [ ]:
import shutil
rag.clear()
shutil.rmtree('./output', ignore_errors=True)

print('=' * 50)
print('  SightRAG v0.3 Demo Complete!')
print('=' * 50)
print()
print('  Tested:')
print('  - Image folder indexing')
print('  - Text + reference queries')
print('  - rag.show() visualization')
print('  - Grounding DINO (any domain)')
print('  - Re-ranking (better accuracy)')
print()
print('  github.com/VK-Ant/sightrag')
print('  pip install sightrag')